In [1]:
import mne
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import regex as reg
from sklearn.pipeline import Pipeline
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [3]:
import warnings

mne.set_log_level('ERROR')   # silence MNE

warnings.filterwarnings("ignore")  # silence warnings

In [13]:
import mne
import numpy as np
import os
class preprocessing_pipeline:
    def __init__(self, filename, *channel_tuple, l_freq=8.0, h_freq=30.0, notch_freq=50.0, fs=500.0, time_window=0.8):
        self.filename = filename
        self.l_freq = l_freq
        self.h_freq = h_freq
        self.notch_freq = notch_freq
        self.time_window = time_window
        self.fs = fs
        self.active_channels = channel_tuple[0]
        self.bad_channels = channel_tuple[1]

        # Load and clean
        self.raw = self.file_process()
        
        # Get annotations as a DataFrame for easy time-math
        self.annotations = self.raw.annotations

    def file_process(self):
        raw = mne.io.read_raw_egi(self.filename, preload=True)
        if 'VREF' in raw.ch_names:
            raw.drop_channels(['VREF'])
        
        raw.pick('eeg')
        if self.bad_channels:
            raw.drop_channels([ch for ch in self.bad_channels if ch in raw.ch_names])
            
        raw.set_eeg_reference('average', projection=False)
        raw.filter(l_freq=self.l_freq, h_freq=self.h_freq, phase='zero')
        raw.notch_filter(freqs=self.notch_freq)
        return raw
    
    def extracting_data(self, start_offset=0.0, end_offset=0.0, overlap_factor=0.8):
        classes = ['BA', 'BY', 'DO', 'MO', 'SI']
        # Changed from flat list to a dictionary grouped by class
        trial_groups = {cls: [] for cls in classes} 

        for cls in classes:
            starts = [ann['onset'] for ann in self.annotations if ann['description'] == f'OS{cls}']
            ends   = [ann['onset'] for ann in self.annotations if ann['description'] == f'OE{cls}']

            for start, end in zip(starts, ends):
                segment = self.raw.copy().crop(tmin=start+start_offset, tmax=end+end_offset)
                data = segment.get_data(picks='eeg').astype(np.float32)

                window_samples = int(self.time_window * self.fs)
                step_samples = int(window_samples * (1-overlap_factor))
                
                total_samples = data.shape[1]
                this_trial_windows = []
                
                for start_pt in range(0, total_samples - window_samples + 1, step_samples):
                    chunk = data[:, start_pt:start_pt + window_samples]
                    this_trial_windows.append(chunk)

                if this_trial_windows:
                    # Store as a tuple: (Array of Windows, Label)
                    X_windows = np.stack(this_trial_windows, axis=0)
                    y_windows = np.full(X_windows.shape[0], label_dict[f'OS{cls}'])
                    trial_groups[cls].append((X_windows, y_windows))

        return trial_groups

## Notes:
### * The PLV is computed on the whole segment
### * Node features are based on the chunks but will have edges based on the whole segment

## PLV computation

In [5]:
def compute_plv(segment: np.ndarray) -> np.ndarray:
    """
    Compute Phase Locking Value matrix for one segment(whole length of the event).
    
    Args:
        segment: (n_channels, n_samples)  float32/64
    Returns:
        plv_matrix: (n_channels, n_channels)  values in [0, 1]
    """
    n_ch = segment.shape[0]
    # Analytic signal via Hilbert → instantaneous phase
    from scipy.signal import hilbert
    phases = np.angle(hilbert(segment, axis=1))   # (n_ch, n_samples)
    
    plv = np.zeros((n_ch, n_ch), dtype=np.float32)
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            delta_phi = phases[i] - phases[j]
            plv_val = np.abs(np.mean(np.exp(1j * delta_phi)))
            plv[i, j] = plv_val
            plv[j, i] = plv_val
    np.fill_diagonal(plv, 1.0)
    return plv

In [6]:
def plv_to_edge_index(plv_matrix: np.ndarray, 
                       threshold: float = 0.3):
    """
    Threshold PLV matrix → sparse edge_index + edge_attr.
    
    Args:
        plv_matrix: (n_channels, n_channels)
        threshold:  keep edges where PLV > threshold
    Returns:
        edge_index: (2, n_edges)  long tensor
        edge_attr:  (n_edges, 1)  float tensor  (PLV weights)
    """
    rows, cols = np.where(plv_matrix > threshold)
    # Remove self-loops (diagonal already handled by fill_diagonal but just in case)
    mask = rows != cols
    rows, cols = rows[mask], cols[mask]
    
    edge_index = torch.tensor(np.stack([rows, cols], axis=0), dtype=torch.long)
    edge_attr  = torch.tensor(plv_matrix[rows, cols], dtype=torch.float32).unsqueeze(1)
    return edge_index, edge_attr

In [7]:
def extract_node_features(segment: np.ndarray, 
                           sfreq: float = 500.0,
                           window_size: float = 0.6) -> np.ndarray:
    """
    For each channel (node), compute band-power features over 0.6s windows
    then concatenate as the node feature vector.
    
    Feature per window: [mean_power, alpha_power, beta_power, std]
    Final node feature: concatenation across all windows in the 4s segment.
    
    Args:
        segment:     (n_channels, n_samples)
        sfreq:       sampling frequency
        window_size: in seconds
    Returns:
        node_features: (n_channels, n_features)
    """
    from scipy.signal import welch
    
    n_ch, n_samples = segment.shape
    win_samples = int(window_size * sfreq)
    n_windows = n_samples // win_samples
    
    all_features = []
    
    for w in range(n_windows):
        s = w * win_samples
        e = s + win_samples
        chunk = segment[:, s:e]                      # (n_ch, win_samples)
        
        # Welch PSD per channel for this window
        freqs, psd = welch(chunk, fs=sfreq, nperseg=min(win_samples, 128), axis=1)
        
        alpha_mask = (freqs >= 8)  & (freqs <= 13)
        beta_mask  = (freqs >= 14) & (freqs <= 26)
        
        alpha_power = psd[:, alpha_mask].mean(axis=1)   # (n_ch,)
        beta_power  = psd[:, beta_mask].mean(axis=1)    # (n_ch,)
        mean_power  = psd.mean(axis=1)                  # (n_ch,)
        std_amp     = chunk.std(axis=1)                 # (n_ch,)
        
        # Stack → (n_ch, 4)
        window_feat = np.stack([mean_power, alpha_power, beta_power, std_amp], axis=1)
        all_features.append(window_feat)
    
    # Concatenate across windows → (n_ch, 4 * n_windows)
    node_features = np.concatenate(all_features, axis=1).astype(np.float32)
    return node_features


In [8]:
def build_graph_dataset(pipeline, 
                         plv_threshold: float = 0.3,
                         label_dict: dict = None) -> list:
    """
    Converts preprocessed pipeline output into a list of PyG Data objects.
    One graph per 4s OS→OE segment.
    
    Args:
        pipeline:      fitted preprocessing_pipeline instance
        plv_threshold: edge inclusion threshold
        label_dict:    e.g. {'OSBA': 0, 'OSBY': 1, ...}
    Returns:
        graph_list: list of torch_geometric.data.Data
    """
    classes = ['BA', 'BY', 'DO', 'MO', 'SI']
    
    if label_dict is None:
        label_dict = {f'OS{cls}': i for i, cls in enumerate(classes)}
    
    graph_list = []
    
    for cls in classes:
        starts = [ann['onset'] for ann in pipeline.annotations 
                  if ann['description'] == f'OS{cls}']
        ends   = [ann['onset'] for ann in pipeline.annotations 
                  if ann['description'] == f'OE{cls}']
        
        for start, end in zip(starts, ends):
            segment = pipeline.raw.copy().crop(tmin=start, tmax=end)
            data_arr = segment.get_data(picks='eeg').astype(np.float32)
            # data_arr: (n_channels, n_samples)
            
            # --- Adjacency: PLV over full 4s segment ---
            plv_mat = compute_plv(data_arr)
            edge_index, edge_attr = plv_to_edge_index(plv_mat, threshold=plv_threshold)
            
            # --- Node features: band power over 0.6s windows ---
            node_feat = extract_node_features(data_arr, 
                                               sfreq=pipeline.fs,
                                               window_size=pipeline.time_window)
            # node_feat: (n_channels, n_features)
            
            x = torch.tensor(node_feat, dtype=torch.float32)
            y = torch.tensor([label_dict[f'OS{cls}']], dtype=torch.long)
            
            graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
            graph_list.append(graph)
    
    print(f"  Built {len(graph_list)} graphs | "
          f"Nodes: {graph_list[0].num_nodes} | "
          f"Node feature dim: {graph_list[0].x.shape[1]} | "
          f"Avg edges: {np.mean([g.num_edges for g in graph_list]):.0f}")
    
    return graph_list

In [9]:
class EEG_GAT(nn.Module):
    """
    Graph Attention Network for EEG visual imagery classification.
    
    Architecture:
        GATConv(in  → 64, heads=4)  →  residual proj
        GATConv(256 → 64, heads=4)  →  residual
        GATConv(256 → 64, heads=1)
        GlobalMeanPool
        MLP(64 → 128 → n_classes)
    """
    def __init__(self, in_channels: int, n_classes: int = 5,
                 hidden: int = 64, heads: int = 4, dropout: float = 0.4):
        super().__init__()
        
        self.dropout = dropout
        
        # GAT layers
        self.conv1 = GATConv(in_channels, hidden, heads=heads, 
                              edge_dim=1, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, hidden, heads=heads,
                              edge_dim=1, dropout=dropout)
        self.conv3 = GATConv(hidden * heads, hidden, heads=1,
                              edge_dim=1, dropout=dropout)
        
        # Residual projections (to match dimensions after concatenation)
        self.res1 = nn.Linear(in_channels,    hidden * heads)
        self.res2 = nn.Linear(hidden * heads, hidden * heads)
        self.res3 = nn.Linear(hidden * heads, hidden)
        
        # Batch norms
        self.bn1 = nn.BatchNorm1d(hidden * heads)
        self.bn2 = nn.BatchNorm1d(hidden * heads)
        self.bn3 = nn.BatchNorm1d(hidden)
        
        # Classifier MLP
        self.mlp = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes)
        )
    
    def forward(self, x, edge_index, edge_attr, batch):
        # Layer 1
        h = self.conv1(x, edge_index, edge_attr)
        h = self.bn1(h)
        h = F.elu(h + self.res1(x))
        h = F.dropout(h, p=self.dropout, training=self.training)
        
        # Layer 2
        h2 = self.conv2(h, edge_index, edge_attr)
        h2 = self.bn2(h2)
        h2 = F.elu(h2 + self.res2(h))
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        
        # Layer 3
        h3 = self.conv3(h2, edge_index, edge_attr)
        h3 = self.bn3(h3)
        h3 = F.elu(h3 + self.res3(h2))
        
        # Global pooling: aggregate all node embeddings → graph embedding
        graph_emb = global_mean_pool(h3, batch)   # (batch_size, hidden)
        
        return self.mlp(graph_emb)

In [11]:
def train_gnn(graph_list: list,
              n_classes:  int   = 5,
              epochs:     int   = 100,
              lr:         float = 1e-3,
              batch_size: int   = 8,
              n_folds:    int   = 5,
              device_str: str   = 'auto'):
    """
    Stratified K-Fold cross-validation training loop.
    
    Args:
        graph_list: output of build_graph_dataset()
        n_classes:  number of output classes
        epochs:     training epochs per fold
        lr:         learning rate
        batch_size: graphs per batch
        n_folds:    cross-validation folds
        device_str: 'auto', 'cpu', or 'cuda'
    """
    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
        if device_str == 'auto' else device_str
    )
    print(f"\n Using device: {device}")
    
    labels  = np.array([g.y.item() for g in graph_list])
    in_dim  = graph_list[0].x.shape[1]
    
    skf     = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accs = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(graph_list, labels)):
        print(f"\n{'─'*40}")
        print(f"  Fold {fold+1}/{n_folds}")
        
        train_graphs = [graph_list[i] for i in train_idx]
        val_graphs   = [graph_list[i] for i in val_idx]
        
        train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_graphs,   batch_size=batch_size, shuffle=False)
        
        model     = EEG_GAT(in_channels=in_dim, n_classes=n_classes).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.CrossEntropyLoss()
        
        best_val_acc = 0.0
        for epoch in range(1, epochs + 1):
            tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
            va_loss, va_acc = eval_epoch(model,  val_loader,   criterion, device)
            scheduler.step()
            
            if va_acc > best_val_acc:
                best_val_acc = va_acc
                torch.save(model.state_dict(), f'best_fold{fold+1}.pt')
            
            if epoch % 10 == 0:
                print(f"  Epoch {epoch:3d} | "
                      f"Train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
                      f"Val loss {va_loss:.3f} acc {va_acc:.3f}")
        
        fold_accs.append(best_val_acc)
        print(f"  ✅ Best val acc fold {fold+1}: {best_val_acc:.4f}")
    
    print(f"\n{'═'*40}")
    print(f"  Mean val acc: {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
    return fold_accs

In [14]:
latest_channel_list = [
    # Left sensorimotor area channels
    'E29', 'E30', 'E35', 'E36', 'E41', 'E42',
    # Right sensorimotor area channels
    'E103', 'E104', 'E109', 'E110', 'E115', 'E116',
    # Mid-parietal & bilateral parietal
    'E62', 'E67', 'E72', 'E77'
 ]

bad_channels = ['E48', 'E119', 'E49', 'E113', 'E94', 'E68', 'E23', 'E3', 'E126', 'E127']

channel_tuple = (latest_channel_list, bad_channels)

label_dict = {'OSBA': 0, 'OSBY': 1, 'OSDO': 2, 'OSMO': 3, 'OSSI':4}

directions = ['OSBA', 'OSBY', 'OSDO', 'OSDO','OSSI']  # Left, Right, Up, Down

In [15]:

# Point this to the parent "Data" directory
base_dir = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/Data"

subject_dirs = {}

# 1. Get all items in the Data folder
# 2. Filter for directories that start with 'S'
sub_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and f.startswith('S')]

for folder in sub_folders:
    folder_path = os.path.join(base_dir, folder)
    files = []
    
    # List all .mff files within each subject's folder
    for file_name in os.listdir(folder_path):
        if not file_name.startswith('.') and file_name.endswith('.mff'):
            files.append(file_name)
    
    # Using the actual folder name (e.g., 'S1', 'S113') as the key
    subject_dirs[folder] = files

# Verification
print(f"Found {len(subject_dirs)} subjects.")
print("Subjects identified:", list(subject_dirs.keys()))

Found 13 subjects.
Subjects identified: ['S116', 'S118', 'S5', 'S2', 'S119', 'S117', 'S3', 'S4', 'S1', 'S6', 'S115', 'S113', 'S114']


In [16]:
total_data = {}
test_data = {}
#base_dir = r'D:\0001_AK\KRISHNA\001_EEG_work_recent\extracted_files'
base_dir = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/Data"

for subject, files in subject_dirs.items(): # subject is id, files are the all the files associated with a subject
    print(f"Processing {subject}")
    
    total_data[f"{subject}"] = {} #?
    test_data[f"{subject}"] = {}
    signals = [] #?
    labels = []#?
    signals_test = []
    labels_test = []
    k = 0
    for file_name in files:
        k +=1
        file_path = os.path.join(base_dir,subject, file_name) # grabbing file path, the mff file?
        
        if not file_name.endswith('.mff'):
            print(f"Skipping non-raw file: {file_name}")
            continue
        
        required_parts = ["signal1.bin", "info1.xml"]
        missing_parts = [p for p in required_parts if not os.path.exists(os.path.join(file_path, p))] # wha tis happenign here?
        if missing_parts:
            print(f"Skipping {file_name} due to parts being missing")
            continue
        
        print(f"File is intact: {file_name}\n Beginning extraction...")
        
        # try:
        #     # if k==3 or subject == "S114" and k =2:
        #     if k ==2: # some have only 2 blocks that's why
        #         # we have to some how take one image from each class for the test, and toss the rest of the 
        #         # epochs to the total_data
        #         print("one block is for testing")
        #         processor = preprocessing_pipeline(file_path, *channel_tuple)
        #         X,y = processor.extracting_data()
        #         signals_test.append(X)
        #         labels_test.append(y)
        
        #     else:
        #         print("file path",file_path)
        #         processor = preprocessing_pipeline(file_path, *channel_tuple)
        #         X,y = processor.extracting_data()
        #         signals.append(X)
        #         labels.append(y)
        # except Exception as e:
        #     print(f"Error processing {file_name}: {e}")
        #     continue

        try:
            processor = preprocessing_pipeline(file_path, *channel_tuple)
            # trial_data is now a dict: {'BA': [(win, lab), (win, lab), (win, lab)], ...}
            trial_data = processor.extracting_data()

            if k == 2:
                print(f"Splitting Block {k} into Training and Test...")
                for cls, trials in trial_data.items():
                    # 1. Take the LAST trial (image event) for Testing
                    test_trial_x, test_trial_y = trials.pop() 
                    print(test_trial_x,test_trial_y)
                    signals_test.append(test_trial_x)
                    labels_test.append(test_trial_y)
                    
                    # 2. Put the REMAINING trials from this block into Training
                    for x, y in trials:
                        print(x,y)
                        signals.append(x)
                        labels.append(y)
            else:
                # For Block 1 or 3, just put everything into Training
                for cls, trials in trial_data.items():
                    for x, y in trials:
                        signals.append(x)
                        labels.append(y)

        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            continue

    
    total_data[f"{subject}"]['data'] = np.concatenate(signals, axis=0)
    total_data[f"{subject}"]['labels'] = np.concatenate(labels, axis=0)   
    
    test_data[f"{subject}"]['data'] = np.concatenate(signals_test, axis=0)
    test_data[f"{subject}"]['labels'] = np.concatenate(labels_test, axis=0)  

    

Processing S116
File is intact: VI_S6_S1_B3__20251116_110436.mff
 Beginning extraction...
File is intact: VI_S6_S1_B1__20251116_104819.mff
 Beginning extraction...
Splitting Block 2 into Training and Test...
[[[ 1.02506738e-06  8.29561884e-07  6.63615594e-07 ... -1.51045551e-05
   -1.56989008e-05 -1.54471818e-05]
  [ 1.04179162e-06  1.07320852e-06  1.12017176e-06 ... -1.44128990e-05
   -1.31453735e-05 -1.10219335e-05]
  [-4.42424550e-07 -2.14146894e-07  1.86084961e-08 ... -2.92639402e-06
   -1.63633797e-06 -2.01035235e-07]
  ...
  [ 1.17230719e-07  5.98175518e-08 -3.61366403e-08 ... -4.57141505e-06
   -3.64013272e-06 -2.48752804e-06]
  [ 1.81712358e-06  1.38785163e-06  1.00100317e-06 ... -1.12351345e-05
   -1.25833531e-05 -1.33069398e-05]
  [ 1.04390801e-06  5.97866915e-07  2.08758024e-07 ...  1.61482367e-05
    1.28131760e-05  9.45443026e-06]]

 [[-4.83228314e-06 -3.42950921e-06 -1.93419942e-06 ... -1.13551346e-06
    4.74253056e-07  1.69521445e-06]
  [-5.64723041e-06 -4.14987699e-06 

In [ ]:

# --- 2. Build graph dataset ---
label_dict = {f'OS{cls}': i for i, cls in enumerate(['BA', 'BY', 'DO', 'MO', 'SI'])}
graphs = build_graph_dataset(pipeline, plv_threshold=0.3, label_dict=label_dict)

# --- 3. Train ---
fold_accuracies = train_gnn(graphs, n_classes=5, epochs=100, lr=1e-3, batch_size=8)